In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split

os.environ.setdefault("PGDATABASE", "DW_Energia_ML")
os.environ.setdefault("PGUSER", "sa")
os.environ.setdefault("PGPASSWORD", "progra")
os.environ.setdefault("PGHOST", "localhost")
os.environ.setdefault("PGPORT", "5433")

from modelos.ModelosML import ModelosML


## Cargar la vista modelo desde la BD

In [2]:
modelo_prediccion_precio_mes = ModelosML()
modelo_prediccion_precio_mes.sql_view_to_df("prediccion_precio_mes", schema="Fact_Dim")\
                   .separar_columnas_texto_numericas()

print("Vista prediccion_precio_mes:", modelo_prediccion_precio_mes.df.shape)
print("Columnas con texto:", len(modelo_prediccion_precio_mes.columnas_texto))
print(modelo_prediccion_precio_mes.df_texto)
print("Columnas numericas:", len(modelo_prediccion_precio_mes.columnas_numericas))
print(modelo_prediccion_precio_mes.df_numericas)

display(modelo_prediccion_precio_mes.df.head())


Vista prediccion_precio_mes: (4608, 40)
Columnas con texto: 12
      anio nombre_mes nombre_empresa                    sistema  \
0     2025  Diciembre           CNFL  DISTRIBUCION | GENERACION   
1     2025  Diciembre           CNFL  DISTRIBUCION | GENERACION   
2     2025  Diciembre           CNFL  DISTRIBUCION | GENERACION   
3     2025  Diciembre           CNFL  DISTRIBUCION | GENERACION   
4     2025  Diciembre           CNFL  DISTRIBUCION | GENERACION   
...    ...        ...            ...                        ...   
4603  2018      Enero          JASEC  DISTRIBUCION | GENERACION   
4604  2018      Enero          JASEC  DISTRIBUCION | GENERACION   
4605  2018      Enero          JASEC  DISTRIBUCION | GENERACION   
4606  2018      Enero          JASEC  DISTRIBUCION | GENERACION   
4607  2018      Enero          JASEC  DISTRIBUCION | GENERACION   

                                                 tarifa  \
0     ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...   
1     ALUMBRAD

,anio,mes,trimestre,nombre_mes,nombre_empresa,sistema,tarifa,descripcion_tarifa,id_objecto,central_electrica,...,t2m_min,cloud_od,gwetroot,ts,prectotcorr,allsky_sfc_sw_dwn,ps,t2mwet,allsky_sfc_sw_diff,allsky_sfc_lw_dwn
0,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,8,P.E VALLE CENTRAL,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
1,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,12,VENTANAS,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
2,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,13,ANONOS,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
3,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,14,RÍO SEGUNDO,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
4,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,16,BRASIL,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684


In [3]:
x,y = modelo_prediccion_precio_mes.df.shape
print(x,y)
print(x*y)
modelo_prediccion_precio_mes.df.info()


4608 40
184320
<class 'pandas.DataFrame'>
RangeIndex: 4608 entries, 0 to 4607
Data columns (total 40 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   anio                             4608 non-null   str    
 1   mes                              4608 non-null   int64  
 2   trimestre                        4608 non-null   int64  
 3   nombre_mes                       4608 non-null   str    
 4   nombre_empresa                   4608 non-null   str    
 5   sistema                          4608 non-null   str    
 6   tarifa                           4608 non-null   str    
 7   descripcion_tarifa               4608 non-null   str    
 8   id_objecto                       4608 non-null   int64  
 9   central_electrica                4608 non-null   str    
 10  operador                         4608 non-null   str    
 11  fuente                           4608 non-null   str    
 12  provincia       

In [4]:
modelo_prediccion_precio_mes.rm_col('id_objecto', axis=1)
modelo_prediccion_precio_mes.df


,anio,mes,trimestre,nombre_mes,nombre_empresa,sistema,tarifa,descripcion_tarifa,central_electrica,operador,...,t2m_min,cloud_od,gwetroot,ts,prectotcorr,allsky_sfc_sw_dwn,ps,t2mwet,allsky_sfc_sw_diff,allsky_sfc_lw_dwn
0,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,P.E VALLE CENTRAL,COMPAÑIA NACIONAL DE FUERZA Y LUZ S.A.,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
1,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,VENTANAS,COMPAÑIA NACIONAL DE FUERZA Y LUZ S.A.,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
2,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ANONOS,COMPAÑIA NACIONAL DE FUERZA Y LUZ S.A.,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
3,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,RÍO SEGUNDO,COMPAÑIA NACIONAL DE FUERZA Y LUZ S.A.,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
4,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,BRASIL,COMPAÑIA NACIONAL DE FUERZA Y LUZ S.A.,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4603,2018,1,1,Enero,JASEC,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,BARRO MORADO,JUNTA ADMINISTRATIVA DEL SERVICIO ELECTRICO MU...,...,15.680000,7.980000,0.940000,18.950000,11.850000,3.888500,89.730000,18.300000,1.741400,8.745800
4604,2018,1,1,Enero,JASEC,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,TUIS,JUNTA ADMINISTRATIVA DEL SERVICIO ELECTRICO MU...,...,15.680000,7.980000,0.940000,18.950000,11.850000,3.888500,89.730000,18.300000,1.741400,8.745800
4605,2018,1,1,Enero,JASEC,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,BIRRIS I,JUNTA ADMINISTRATIVA DEL SERVICIO ELECTRICO MU...,...,15.680000,7.980000,0.940000,18.950000,11.850000,3.888500,89.730000,18.300000,1.741400,8.745800
4606,2018,1,1,Enero,JASEC,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,BIRRIS III,JUNTA ADMINISTRATIVA DEL SERVICIO ELECTRICO MU...,...,15.680000,7.980000,0.940000,18.950000,11.850000,3.888500,89.730000,18.300000,1.741400,8.745800


In [5]:
modelo_prediccion_consumo_empresa = ModelosML()
modelo_prediccion_consumo_empresa.sql_view_to_df("prediccion_consumo_empresa", schema="Fact_Dim")\
                     .separar_columnas_texto_numericas()

print("Vista prediccion_consumo_empresa:", modelo_prediccion_consumo_empresa.df.shape)
print("Columnas con texto:", len(modelo_prediccion_consumo_empresa.columnas_texto))
print(modelo_prediccion_consumo_empresa.df_texto)
print("Columnas numericas:", len(modelo_prediccion_consumo_empresa.columnas_numericas))
print(modelo_prediccion_consumo_empresa.df_numericas)

display(modelo_prediccion_consumo_empresa.df.head())


Vista prediccion_consumo_empresa: (5609, 19)
Columnas con texto: 5
      anio nombre_mes     nombre_empresa                 tarifa_electricidad  \
0     2018      Enero               CNFL                   ALUMBRADO PÚBLICO   
1     2018      Enero               CNFL               COMERCIOS Y SERVICIOS   
2     2018      Enero               CNFL                          INDUSTRIAL   
3     2018      Enero               CNFL                     MEDIA TENSIÓN A   
4     2018      Enero               CNFL                        PREFERENCIAL   
...    ...        ...                ...                                 ...   
5604  2025  Diciembre              JASEC                     MEDIA TENSIÓN B   
5605  2025  Diciembre              JASEC                        PREFERENCIAL   
5606  2025  Diciembre              JASEC                         RESIDENCIAL   
5607  2025  Diciembre              JASEC  VENTAS AL SERVICIO DE DISTRIBUCIÓN   
5608  2025  Diciembre  USUARIOS DIRECTOS             

,fecha_mes,fecha_anio,anio,mes,trimestre,nombre_mes,nombre_empresa,tarifa_electricidad,sistema,empresa_mes_abonados_total,empresa_mes_tarifa_consumo_energia_kwh_total,empresa_mes_tarifa_consumo_energia_kwh_promedio_por_abonado,empresa_mes_tarifa_ingreso_sin_cvg_crc_total,empresa_mes_tarifa_ingreso_con_cvg_crc_total,empresa_mes_tarifa_pago_sin_cvg_crc_promedio_por_abonado,empresa_mes_tarifa_pago_con_cvg_crc_promedio_por_abonado,empresa_mes_tarifa_precio_medio_sin_cvg_crc_kwh_promedio,empresa_mes_tarifa_precio_medio_con_cvg_crc_kwh_promedio,empresa_mes_cantidad_centrales
0,2018-01-01,2018-01-01,2018,1,1,Enero,CNFL,ALUMBRADO PÚBLICO,Distribucion,None,7720071.470000,None,478907235.910000,478907235.910000,None,None,62.0300000000000000,62.0300000000000000,9
1,2018-01-01,2018-01-01,2018,1,1,Enero,CNFL,COMERCIOS Y SERVICIOS,Distribucion,69844.000000,89338342.010000,1279.1126225588454270,10038033115.000000,10071190040.000000,143720.765062138480,144195.493385258576,112.3600000000000000,112.7300000000000000,9
2,2018-01-01,2018-01-01,2018,1,1,Enero,CNFL,INDUSTRIAL,Distribucion,1498.000000,5285041.000000,3528.0647530040053405,656039304.000000,658675750.000000,437943.460614152203,439703.437917222964,124.1300000000000000,124.6300000000000000,9
3,2018-01-01,2018-01-01,2018,1,1,Enero,CNFL,MEDIA TENSIÓN A,Distribucion,287.000000,58184898.000000,202734.836236933798,4776178070.000000,4870294175.000000,16641735.435540069686,16969666.114982578397,82.0900000000000000,83.7000000000000000,9
4,2018-01-01,2018-01-01,2018,1,1,Enero,CNFL,PREFERENCIAL,Distribucion,2521.000000,8253336.000000,3273.8341927806426021,542273251.000000,545827305.000000,215102.439904799683,216512.219357397858,65.7000000000000000,66.1300000000000000,9


In [6]:
x,y = modelo_prediccion_consumo_empresa.df.shape
print(x,y)
print(x*y)
modelo_prediccion_consumo_empresa.df.info()


5609 19
106571
<class 'pandas.DataFrame'>
RangeIndex: 5609 entries, 0 to 5608
Data columns (total 19 columns):
 #   Column                                                       Non-Null Count  Dtype 
---  ------                                                       --------------  ----- 
 0   fecha_mes                                                    5609 non-null   object
 1   fecha_anio                                                   5609 non-null   object
 2   anio                                                         5609 non-null   str   
 3   mes                                                          5609 non-null   int64 
 4   trimestre                                                    5609 non-null   int64 
 5   nombre_mes                                                   5609 non-null   str   
 6   nombre_empresa                                               5609 non-null   str   
 7   tarifa_electricidad                                          5609 non-null   str   

## Partición 70% entrenamiento, 15% validación, 15% test

In [7]:
modelo_prediccion_precio_mes.split_datos(
    columna_homogenea="mes",
    proporcion_train=0.70,
    proporcion_test=0.30,
    val=True,
)

modelo_prediccion_consumo_empresa.split_datos(
    columna_homogenea="mes",
    proporcion_train=0.70,
    proporcion_test=0.30,
    val=True,
)

print("Vista modelo - train/validacion/test:")
print(modelo_prediccion_precio_mes.df_train.shape)
print(modelo_prediccion_precio_mes.df_validacion.shape)
print(modelo_prediccion_precio_mes.df_test.shape)

print("Vista tarifas - train/validacion/test:")
print(modelo_prediccion_consumo_empresa.df_train.shape)
print(modelo_prediccion_consumo_empresa.df_validacion.shape)
print(modelo_prediccion_consumo_empresa.df_test.shape)


Vista modelo - train/validacion/test:
(3225, 39)
(691, 39)
(692, 39)
Vista tarifas - train/validacion/test:
(3926, 19)
(841, 19)
(842, 19)


## Entrenar

In [8]:
target_precio = "empresa_mes_tarifa_crc_promedio"
columnas_precio = None

target_consumo = "empresa_mes_tarifa_consumo_energia_kwh_total"
columnas_consumo = None

modelo_prediccion_precio_mes.entrenar_modelos(
    columna_objetivo=target_precio,
    columnas_entrada=columnas_precio,
    modelos_usar=None,
    cv=5,
)

modelo_prediccion_consumo_empresa.entrenar_modelos(
    columna_objetivo=target_consumo,
    columnas_entrada=columnas_consumo,
    modelos_usar=None,
    cv=5,
)

print("Columnas precio:", modelo_prediccion_precio_mes.columnas_entrada)
print("Columnas consumo:", modelo_prediccion_consumo_empresa.columnas_entrada)
print("Modelos precio:", modelo_prediccion_precio_mes.modelos.keys())
print("Modelos consumo:", modelo_prediccion_consumo_empresa.modelos.keys())


Columnas precio: ['anio', 'mes', 'trimestre', 'nombre_mes', 'nombre_empresa', 'sistema', 'tarifa', 'descripcion_tarifa', 'central_electrica', 'operador', 'fuente', 'provincia', 'canton', 'distrito', 'codigo_dta', 'coordenada_x', 'coordenada_y', 'abonados', 'ventas_por_mes', 'ingreso_sin_cvg', 'ingreso_con_cvg', 'precio_medio_sin_cvg', 'precio_medio_con_cvg', 't2m', 'ws10m', 'cloud_amt', 'rh2m', 't2m_max', 't2m_min', 'cloud_od', 'gwetroot', 'ts', 'prectotcorr', 'allsky_sfc_sw_dwn', 'ps', 't2mwet', 'allsky_sfc_sw_diff', 'allsky_sfc_lw_dwn']
Columnas consumo: ['fecha_mes', 'fecha_anio', 'anio', 'mes', 'trimestre', 'nombre_mes', 'nombre_empresa', 'tarifa_electricidad', 'sistema', 'empresa_mes_abonados_total', 'empresa_mes_tarifa_consumo_energia_kwh_promedio_por_abonado', 'empresa_mes_tarifa_ingreso_sin_cvg_crc_total', 'empresa_mes_tarifa_ingreso_con_cvg_crc_total', 'empresa_mes_tarifa_pago_sin_cvg_crc_promedio_por_abonado', 'empresa_mes_tarifa_pago_con_cvg_crc_promedio_por_abonado', 'empre

## Comparacion simple de algoritmos

Se comparan Regresion Lineal (RL) y Random Forest (RF) usando validacion y prueba. El mejor queda en memoria en `modelo`, pero el guardado lo decides despues de revisar los porcentajes.


In [9]:
comparacion_precio = modelo_prediccion_precio_mes.comparar_modelos()
comparacion_consumo = modelo_prediccion_consumo_empresa.comparar_modelos()

print("Precio electricidad: precision por algoritmo")
display(comparacion_precio)
print("Mejor modelo precio en memoria:", modelo_prediccion_precio_mes.mejor_modelo_nombre)

print("Consumo energia: precision por algoritmo")
display(comparacion_consumo)
print("Mejor modelo consumo en memoria:", modelo_prediccion_consumo_empresa.mejor_modelo_nombre)

validacion_precio_mejor = modelo_prediccion_precio_mes.evaluar_validacion(
    modelo=modelo_prediccion_precio_mes.mejor_modelo_nombre
)
validacion_consumo_mejor = modelo_prediccion_consumo_empresa.evaluar_validacion(
    modelo=modelo_prediccion_consumo_empresa.mejor_modelo_nombre
)

def salida_simple_validacion(df, filas=10):
    return pd.DataFrame({
        "real": df["real"].round(2),
        "prediccion": df["prediccion"].round(2),
        "diferencia": df["error"].round(2),
        "error_%": df["error_porcentaje"].round(2),
        "acierto_%": df["acierto_porcentaje"].round(2),
    }).head(filas)

print("Precio electricidad: validacion real vs prediccion del mejor modelo")
display(salida_simple_validacion(validacion_precio_mejor))

print("Consumo energia: validacion real vs prediccion del mejor modelo")
display(salida_simple_validacion(validacion_consumo_mejor))


Precio electricidad: precision por algoritmo


,modelo,acierto_validacion_%,error_validacion_%,acierto_prueba_%,error_prueba_%,r2_validacion,r2_prueba
0,RF,99.49,12.78,99.67,2.58,-12.2498,0.2933
1,RL,53.40,105.10,53.32,101.55,-7.7083,-3.4987


Mejor modelo precio en memoria: RF
Consumo energia: precision por algoritmo


,modelo,acierto_validacion_%,error_validacion_%,acierto_prueba_%,error_prueba_%,r2_validacion,r2_prueba
0,RF,97.33,4.01,97.35,8.679300e+02,0.9992,0.9991
1,RL,51.11,2412.40,52.10,7.403421e+08,0.9785,-10.0209


Mejor modelo consumo en memoria: RF
Precio electricidad: validacion real vs prediccion del mejor modelo


,real,prediccion,diferencia,error_%,acierto_%
0,17407.00,18016.67,-609.68,3.50,96.50
1,1649.08,1649.08,-0.00,0.00,100.00
2,1827.94,1827.94,-0.00,0.00,100.00
3,1648.92,1648.92,-0.00,0.00,100.00
4,1511.09,1511.09,-0.00,0.00,100.00
5,1483.61,1483.61,-0.00,0.00,100.00
6,1710.84,1710.84,0.00,0.00,100.00
7,25795.87,25774.16,21.71,0.08,99.92
8,20828.52,20828.52,-0.00,0.00,100.00
9,1387.74,1387.74,-0.00,0.00,100.00


Consumo energia: validacion real vs prediccion del mejor modelo


,real,prediccion,diferencia,error_%,acierto_%
0,8.462826e+05,8.546345e+05,-8351.92,0.99,99.01
1,3.437335e+07,3.438857e+07,-15220.21,0.04,99.96
2,3.006816e+08,2.953967e+08,5284950.09,1.76,98.24
3,5.592464e+07,5.558059e+07,344055.25,0.62,99.38
4,1.794431e+06,1.783706e+06,10725.14,0.60,99.40
5,2.092365e+06,2.625287e+06,-532921.70,25.47,74.53
6,3.308853e+06,3.320618e+06,-11765.38,0.36,99.64
7,8.230118e+05,8.141044e+05,8907.35,1.08,98.92
8,5.335190e+07,5.487589e+07,-1523989.19,2.86,97.14
9,1.817052e+07,1.816100e+07,9514.17,0.05,99.95


## Guardar ambos modelos en `src/modelos`

In [10]:
modelo_prediccion_precio_mes.save_model(
    "modelo_prediccion_precio_mes",
    metadata={
        "columnas_modelo": modelo_prediccion_precio_mes.columnas_modelo,
        "columnas_entrada": modelo_prediccion_precio_mes.columnas_entrada,
        "valores_relleno": modelo_prediccion_precio_mes.valores_relleno,
        "target": modelo_prediccion_precio_mes.target,
        "metricas": modelo_prediccion_precio_mes.metricas,
    },
)

modelo_prediccion_consumo_empresa.save_model(
    "modelo_prediccion_consumo_empresa",
    metadata={
        "columnas_modelo": modelo_prediccion_consumo_empresa.columnas_modelo,
        "columnas_entrada": modelo_prediccion_consumo_empresa.columnas_entrada,
        "valores_relleno": modelo_prediccion_consumo_empresa.valores_relleno,
        "target": modelo_prediccion_consumo_empresa.target,
        "metricas": modelo_prediccion_consumo_empresa.metricas,
    },
    chain=False
)


{'ok': True,
 'ruta': 'C:\\Users\\d3smo\\Desktop\\CUC\\Progra 2\\Proyectos\\Proyecto_final_Colab\\Prediccion_Consumo_Energia_Costa_Rica\\src\\modelos\\modelo_prediccion_consumo_empresa.joblib',
 'payload': {'modelo': {'RF': RandomForestRegressor(random_state=42)},
  'columnas_modelo': ['fecha_mes_2018-01-01',
   'fecha_mes_2018-02-01',
   'fecha_mes_2018-03-01',
   'fecha_mes_2018-04-01',
   'fecha_mes_2018-05-01',
   'fecha_mes_2018-06-01',
   'fecha_mes_2018-07-01',
   'fecha_mes_2018-08-01',
   'fecha_mes_2018-09-01',
   'fecha_mes_2018-10-01',
   'fecha_mes_2018-11-01',
   'fecha_mes_2018-12-01',
   'fecha_mes_2019-01-01',
   'fecha_mes_2019-02-01',
   'fecha_mes_2019-03-01',
   'fecha_mes_2019-04-01',
   'fecha_mes_2019-05-01',
   'fecha_mes_2019-06-01',
   'fecha_mes_2019-07-01',
   'fecha_mes_2019-08-01',
   'fecha_mes_2019-09-01',
   'fecha_mes_2019-10-01',
   'fecha_mes_2019-11-01',
   'fecha_mes_2019-12-01',
   'fecha_mes_2020-01-01',
   'fecha_mes_2020-02-01',
   'fecha_mes_

## Probar carga y consumo de modelos guardados

Esta seccion valida que `cargar_modelo()` restaura los modelos guardados y que `predecir()` puede recibir solo los datos ingresados para la prediccion. Las demas variables del entrenamiento se completan con valores aprendidos del train.


In [11]:
modelo_precio_guardado = ModelosML()
modelo_precio_guardado.cargar_modelo("modelo_prediccion_precio_mes")

datos_precio = {
    "anio": 2026,
    "mes": 1,
    "trimestre": 1,
    "nombre_empresa": "ICE",
    "provincia": "CARTAGO",
    "canton": "CARTAGO",
    "distrito": "ORIENTAL",
    "coordenada_x": -83.9194,
    "coordenada_y": 9.8644,
}

prediccion_precio = modelo_precio_guardado.predecir(datos_precio, modelo="RF")
display(prediccion_precio)

modelo_consumo_guardado = ModelosML()
modelo_consumo_guardado.cargar_modelo("modelo_prediccion_consumo_empresa")

datos_consumo = {
    "anio": 2026,
    "mes": 1,
    "trimestre": 1,
    "nombre_empresa": "ICE",
    "tarifa_electricidad": "RESIDENCIAL",
}

prediccion_consumo = modelo_consumo_guardado.predecir(datos_consumo, modelo="RF")
display(prediccion_consumo)

print("Modelos precio cargados:", modelo_precio_guardado.modelos.keys())
print("Modelos consumo cargados:", modelo_consumo_guardado.modelos.keys())
print("Columnas precio usadas:", modelo_precio_guardado.columnas_entrada)
print("Columnas consumo usadas:", modelo_consumo_guardado.columnas_entrada)


,anio,mes,trimestre,nombre_empresa,provincia,canton,distrito,coordenada_x,coordenada_y,prediccion
0,2026,1,1,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,1691.456944


,anio,mes,trimestre,nombre_empresa,tarifa_electricidad,prediccion
0,2026,1,1,ICE,RESIDENCIAL,1.233784e+06


Modelos precio cargados: dict_keys(['RF'])
Modelos consumo cargados: dict_keys(['RF'])
Columnas precio usadas: ['anio', 'mes', 'trimestre', 'nombre_mes', 'nombre_empresa', 'sistema', 'tarifa', 'descripcion_tarifa', 'central_electrica', 'operador', 'fuente', 'provincia', 'canton', 'distrito', 'codigo_dta', 'coordenada_x', 'coordenada_y', 'abonados', 'ventas_por_mes', 'ingreso_sin_cvg', 'ingreso_con_cvg', 'precio_medio_sin_cvg', 'precio_medio_con_cvg', 't2m', 'ws10m', 'cloud_amt', 'rh2m', 't2m_max', 't2m_min', 'cloud_od', 'gwetroot', 'ts', 'prectotcorr', 'allsky_sfc_sw_dwn', 'ps', 't2mwet', 'allsky_sfc_sw_diff', 'allsky_sfc_lw_dwn']
Columnas consumo usadas: ['fecha_mes', 'fecha_anio', 'anio', 'mes', 'trimestre', 'nombre_mes', 'nombre_empresa', 'tarifa_electricidad', 'sistema', 'empresa_mes_abonados_total', 'empresa_mes_tarifa_consumo_energia_kwh_promedio_por_abonado', 'empresa_mes_tarifa_ingreso_sin_cvg_crc_total', 'empresa_mes_tarifa_ingreso_con_cvg_crc_total', 'empresa_mes_tarifa_pago_

## Generar CSV con predicciones de los proximos 12 meses

Se usan los mismos datos base de prediccion y se proyectan los siguientes 12 meses con cada algoritmo disponible.

In [12]:
def proximos_12_meses(anio_inicio, mes_inicio):
    meses = []
    for i in range(12):
        mes_total = mes_inicio + i
        anio = anio_inicio + (mes_total - 1) // 12
        mes = ((mes_total - 1) % 12) + 1
        trimestre = ((mes - 1) // 3) + 1
        meses.append({"anio": anio, "mes": mes, "trimestre": trimestre})
    return meses

def generar_entradas_12_meses(datos_base, anio_inicio, mes_inicio):
    filas = []
    for fecha in proximos_12_meses(anio_inicio, mes_inicio):
        fila = datos_base.copy()
        fila.update(fecha)
        filas.append(fila)
    return pd.DataFrame(filas)

def predecir_con_cada_modelo(instancia, datos):
    predicciones = {}
    for nombre_modelo in instancia.modelos.keys():
        prediccion = instancia.predecir(
            datos,
            modelo=nombre_modelo,
            incluir_rellenos=False,
        ).copy()
        prediccion = prediccion.rename(columns={"prediccion": instancia.target})
        predicciones[nombre_modelo] = prediccion
    return predicciones

anio_inicio_prediccion = 2027
mes_inicio_prediccion = 1

datos_precio_base = {
    "nombre_empresa": "ICE",
    "provincia": "CARTAGO",
    "canton": "CARTAGO",
    "distrito": "ORIENTAL",
    "coordenada_x": -83.9194,
    "coordenada_y": 9.8644,
}

datos_consumo_base = {
    "anio": anio_inicio_prediccion,
    "mes": mes_inicio_prediccion,
    "trimestre": ((mes_inicio_prediccion - 1) // 3) + 1,
    "nombre_empresa": "ICE",
    "tarifa_electricidad": "RESIDENCIAL",
}

entradas_precio_12_meses = generar_entradas_12_meses(
    datos_precio_base,
    anio_inicio_prediccion,
    mes_inicio_prediccion,
)

entradas_consumo_12_meses = generar_entradas_12_meses(
    datos_consumo_base,
    anio_inicio_prediccion,
    mes_inicio_prediccion,
)

predicciones_precio_12_meses = predecir_con_cada_modelo(
    modelo_prediccion_precio_mes,
    entradas_precio_12_meses,
)

predicciones_consumo_12_meses = predecir_con_cada_modelo(
    modelo_prediccion_consumo_empresa,
    entradas_consumo_12_meses,
)

carpeta_predicciones = modelo_prediccion_precio_mes.BASE_DIR / "data" / "predicciones"
carpeta_predicciones.mkdir(parents=True, exist_ok=True)

archivos_predicciones = {
    "prediccion_precio_mes": predicciones_precio_12_meses,
    "prediccion_consumo_empresa": predicciones_consumo_12_meses,
}

columnas_salida = {
    "prediccion_precio_mes": [
        "anio", "mes", "trimestre", "nombre_empresa", "provincia", "canton",
        "distrito", "coordenada_x", "coordenada_y", modelo_prediccion_precio_mes.target,
    ],
    "prediccion_consumo_empresa": [
        "anio", "mes", "trimestre", "nombre_empresa", "tarifa_electricidad",
        modelo_prediccion_consumo_empresa.target,
    ],
}

rutas_predicciones = []
for nombre_vista, predicciones_modelo in archivos_predicciones.items():
    for nombre_modelo, df_prediccion in predicciones_modelo.items():
        nombre_archivo = f"{nombre_vista}_{nombre_modelo}_proximos_12_meses.csv"
        ruta_archivo = carpeta_predicciones / nombre_archivo
        df_prediccion = df_prediccion[columnas_salida[nombre_vista]]
        df_prediccion.to_csv(ruta_archivo, index=False)
        rutas_predicciones.append(ruta_archivo)

print("CSV generados por modelo:")
for ruta in rutas_predicciones:
    print(ruta)

print("Predicciones precio")
for nombre_modelo, df_prediccion in predicciones_precio_12_meses.items():
    print(nombre_modelo)
    display(df_prediccion)

print("Predicciones consumo")
for nombre_modelo, df_prediccion in predicciones_consumo_12_meses.items():
    print(nombre_modelo)
    display(df_prediccion)


CSV generados por modelo:
C:\Users\d3smo\Desktop\CUC\Progra 2\Proyectos\Proyecto_final_Colab\Prediccion_Consumo_Energia_Costa_Rica\data\predicciones\prediccion_precio_mes_RL_proximos_12_meses.csv
C:\Users\d3smo\Desktop\CUC\Progra 2\Proyectos\Proyecto_final_Colab\Prediccion_Consumo_Energia_Costa_Rica\data\predicciones\prediccion_precio_mes_RF_proximos_12_meses.csv
C:\Users\d3smo\Desktop\CUC\Progra 2\Proyectos\Proyecto_final_Colab\Prediccion_Consumo_Energia_Costa_Rica\data\predicciones\prediccion_consumo_empresa_RL_proximos_12_meses.csv
C:\Users\d3smo\Desktop\CUC\Progra 2\Proyectos\Proyecto_final_Colab\Prediccion_Consumo_Energia_Costa_Rica\data\predicciones\prediccion_consumo_empresa_RF_proximos_12_meses.csv
Predicciones precio
RL


,nombre_empresa,provincia,canton,distrito,coordenada_x,coordenada_y,anio,mes,trimestre,empresa_mes_tarifa_crc_promedio
0,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,1,1,5055.288803
1,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,2,1,5085.064559
2,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,3,1,5114.840315
3,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,4,2,5967.009138
4,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,5,2,5996.784893
5,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,6,2,6026.560649
6,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,7,3,6878.729472
7,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,8,3,6908.505228
8,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,9,3,6938.280984
9,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,10,4,7790.449807


RF


,nombre_empresa,provincia,canton,distrito,coordenada_x,coordenada_y,anio,mes,trimestre,empresa_mes_tarifa_crc_promedio
0,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,1,1,1691.456944
1,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,2,1,1691.456944
2,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,3,1,1691.453094
3,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,4,2,1695.361753
4,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,5,2,1695.349216
5,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,6,2,1694.407275
6,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,7,3,1694.555889
7,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,8,3,1694.565411
8,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,9,3,1694.569622
9,ICE,CARTAGO,CARTAGO,ORIENTAL,-83.9194,9.8644,2027,10,4,1692.987197


Predicciones consumo
RL


,anio,mes,trimestre,nombre_empresa,tarifa_electricidad,empresa_mes_tarifa_consumo_energia_kwh_total
0,2027,1,1,ICE,RESIDENCIAL,-26750.641738
1,2027,2,1,ICE,RESIDENCIAL,8966.880972
2,2027,3,1,ICE,RESIDENCIAL,44684.403682
3,2027,4,2,ICE,RESIDENCIAL,46073.047903
4,2027,5,2,ICE,RESIDENCIAL,81790.570613
5,2027,6,2,ICE,RESIDENCIAL,117508.093323
6,2027,7,3,ICE,RESIDENCIAL,118896.737545
7,2027,8,3,ICE,RESIDENCIAL,154614.260254
8,2027,9,3,ICE,RESIDENCIAL,190331.782964
9,2027,10,4,ICE,RESIDENCIAL,191720.427186


RF


,anio,mes,trimestre,nombre_empresa,tarifa_electricidad,empresa_mes_tarifa_consumo_energia_kwh_total
0,2027,1,1,ICE,RESIDENCIAL,1.233784e+06
1,2027,2,1,ICE,RESIDENCIAL,1.233306e+06
2,2027,3,1,ICE,RESIDENCIAL,1.232755e+06
3,2027,4,2,ICE,RESIDENCIAL,1.232216e+06
4,2027,5,2,ICE,RESIDENCIAL,1.233862e+06
5,2027,6,2,ICE,RESIDENCIAL,1.234037e+06
6,2027,7,3,ICE,RESIDENCIAL,1.234447e+06
7,2027,8,3,ICE,RESIDENCIAL,1.234476e+06
8,2027,9,3,ICE,RESIDENCIAL,1.234160e+06
9,2027,10,4,ICE,RESIDENCIAL,1.234971e+06
